# Descriptive Analysis

Phase 1 capstone. Five zones (BE, FI, SG, US-MIDA-PJM, US-NY-NYIS), hourly data 2021-01 through 2025-12. Read-only over `data/raw/`; produces figures bound for the report's Data section and the per-region heterogeneity atlas (Contribution 3).

## 1. Setup

In [ ]:
from __future__ import annotations

from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import carbon_forecast
from carbon_forecast.data.storage import read_parquet
from carbon_forecast.data.zones import ZONES
from carbon_forecast.plotting.config import (
    ENERGY_PALETTE,
    ENERGY_SOURCE_ORDER,
    PLOT_H,
    PLOT_W,
    REGIONAL_PALETTE,
    apply_defaults,
    style_fig,
)

apply_defaults()

PROJECT_ROOT = Path(carbon_forecast.__file__).resolve().parents[2]
DATA_ROOT = PROJECT_ROOT / "data"
ZONE_KEYS = [z.em_key for z in ZONES]

print("Zones    :", ZONE_KEYS)
print("Data root:", DATA_ROOT)


## 2. Long-format master frames

One loader per (zone, endpoint) family. Each concatenates every monthly Parquet on disk into one long DataFrame indexed by UTC time. Cached in memory via `lru_cache` so subsequent cells reuse the same frames.

In [ ]:
def _load_endpoint(zone: str, endpoint: str) -> pd.DataFrame:
    root = DATA_ROOT / "raw/em" / zone / endpoint
    if not root.exists():
        return pd.DataFrame()
    frames = [read_parquet(f) for f in sorted(root.glob("*.parquet"))]
    return pd.concat(frames).sort_index() if frames else pd.DataFrame()


@lru_cache(maxsize=None)
def load_ci_long() -> pd.DataFrame:
    frames = []
    for zone in ZONE_KEYS:
        df = _load_endpoint(zone, "carbon-intensity/past-range")
        if df.empty:
            continue
        df = df.copy()
        df["zone"] = zone
        frames.append(df)
    return pd.concat(frames) if frames else pd.DataFrame()


@lru_cache(maxsize=None)
def load_pb(zone: str) -> pd.DataFrame:
    return _load_endpoint(zone, "power-breakdown/past-range")


@lru_cache(maxsize=None)
def load_flows(zone: str) -> pd.DataFrame:
    return _load_endpoint(zone, "electricity-flows/past-range")


@lru_cache(maxsize=None)
def load_weather(zone: str) -> pd.DataFrame:
    root = DATA_ROOT / "raw/weather" / zone
    if not root.exists():
        return pd.DataFrame()
    frames = [read_parquet(f) for f in sorted(root.glob("*.parquet"))]
    return pd.concat(frames).sort_index() if frames else pd.DataFrame()


def hex_to_rgba(hex_color: str, alpha: float) -> str:
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

In [ ]:
ci_long = load_ci_long()
print("CI long shape :", ci_long.shape)
print("Zones present :", sorted(ci_long["zone"].unique()))
print("Date range    :", ci_long.index.min(), "to", ci_long.index.max())
ci_long.head()

## 3. Analysis helpers

Daily-mean resampling (the granularity for the time-series section), an energy-source to high-level group map (fossil / nuclear / renewable / unknown), and a muted-orange colorscale for the heatmaps.

In [ ]:
# Energy source -> high-level group for the production overview.
SOURCE_GROUP = {
    "nuclear": "nuclear",
    "battery_discharge": "renewable",
    "hydro_discharge": "renewable",
    "hydro": "renewable",
    "wind": "renewable",
    "biomass": "renewable",
    "solar": "renewable",
    "geothermal": "renewable",
    "gas": "fossil",
    "coal": "fossil",
    "oil": "fossil",
    "unknown": "unknown",
}
GROUP_ORDER = ["nuclear", "renewable", "fossil", "unknown"]
GROUP_PALETTE = {
    "nuclear":   "#493C6B",
    "renewable": "#2E9956",
    "fossil":    "#C0504D",
    "unknown":   "#A69C9C",
}

# Muted orange sequential scale for the CI heatmaps.
MUTED_ORANGE = [[0.0, "#FBF1E6"], [0.5, "#E8B07A"], [1.0, "#B5651D"]]


def daily_ci(zone: str) -> pd.Series:
    """Daily-mean carbon intensity for one zone."""
    s = ci_long[ci_long["zone"] == zone]["carbon_intensity_gco2eq_kwh"]
    return s.resample("1D").mean()


def daily_prod_by_source(zone: str) -> pd.DataFrame:
    """Daily-mean production (MW) per source, columns in canonical order."""
    pb = load_pb(zone)
    if pb.empty:
        return pd.DataFrame()
    prod_cols = [c for c in pb.columns if c.startswith("prod_") and c.endswith("_mw")]
    src = pb[prod_cols].copy()
    src.columns = [c[len("prod_"):-len("_mw")] for c in prod_cols]
    daily = src.resample("1D").mean()
    return daily.reindex(columns=[c for c in ENERGY_SOURCE_ORDER if c in daily.columns])


def daily_prod_by_group(zone: str) -> pd.DataFrame:
    """Daily-mean production (MW) aggregated into the four high-level groups."""
    daily = daily_prod_by_source(zone)
    if daily.empty:
        return daily
    grouped = {}
    for grp in GROUP_ORDER:
        cols = [c for c in daily.columns if SOURCE_GROUP.get(c) == grp]
        if cols:
            grouped[grp] = daily[cols].sum(axis=1)
    return pd.DataFrame(grouped)


def daily_net_flow(zone: str) -> pd.Series:
    """Daily-mean net imports (import_total - export_total), MW."""
    pb = load_pb(zone)
    if pb.empty or "import_total_mw" not in pb.columns or "export_total_mw" not in pb.columns:
        return pd.Series(dtype=float)
    net = pb["import_total_mw"].fillna(0.0) - pb["export_total_mw"].fillna(0.0)
    return net.resample("1D").mean()

# Section 1 — Initial time-series exploration

All charts use **daily-mean** values over the full 2021 to 2025 window (about 1,826 points per region), which keeps multi-year trend and seasonality legible while letting bar overlays read cleanly. Regions are stacked vertically (5 rows, shared time axis) for at-a-glance comparison.

### 1. Baseline carbon intensity

Daily-mean carbon intensity per region. The reference series everything else overlays onto.

In [ ]:
fig = make_subplots(
    rows=len(ZONE_KEYS), cols=1, shared_xaxes=True,
    subplot_titles=ZONE_KEYS, vertical_spacing=0.03,
)
for i, zone in enumerate(ZONE_KEYS, start=1):
    s = daily_ci(zone)
    fig.add_trace(
        go.Scatter(x=s.index, y=s.values, mode="lines",
                   line=dict(color=REGIONAL_PALETTE[zone], width=1),
                   name=zone, showlegend=False),
        row=i, col=1,
    )
    fig.update_yaxes(title_text="gCO2eq/kWh", row=i, col=1)
fig.update_xaxes(title_text="Date", row=len(ZONE_KEYS), col=1)
style_fig(fig, "Carbon intensity daily mean by region, 2021 to 2025", height=950)
fig.show()

### 2. Carbon intensity over production mix

Daily-mean production (MW, left axis) stacked into fossil / nuclear / renewable / unknown, with carbon intensity (gCO2eq/kWh, right axis) overlaid as a dark line. Bars are semi-transparent so the CI line reads through. Where renewables and nuclear dominate the stack, CI should sit low; where fossil grows, CI should rise.

In [ ]:
fig = make_subplots(
    rows=len(ZONE_KEYS), cols=1, shared_xaxes=True,
    subplot_titles=ZONE_KEYS, vertical_spacing=0.03,
    specs=[[{"secondary_y": True}] for _ in ZONE_KEYS],
)
seen = set()
for i, zone in enumerate(ZONE_KEYS, start=1):
    grouped = daily_prod_by_group(zone)
    for grp in GROUP_ORDER:
        if grp not in grouped.columns:
            continue
        show = grp not in seen
        seen.add(grp)
        fig.add_trace(
            go.Bar(x=grouped.index, y=grouped[grp], name=grp,
                   marker_color=GROUP_PALETTE[grp], marker_line_width=0,
                   opacity=0.55, legendgroup=grp, showlegend=show),
            row=i, col=1, secondary_y=False,
        )
    ci = daily_ci(zone)
    fig.add_trace(
        go.Scatter(x=ci.index, y=ci.values, mode="lines",
                   line=dict(color="#222222", width=1),
                   name="carbon intensity", legendgroup="ci", showlegend=(i == 1)),
        row=i, col=1, secondary_y=True,
    )
    fig.update_yaxes(title_text="MW", row=i, col=1, secondary_y=False)
    fig.update_yaxes(title_text="gCO2eq/kWh", row=i, col=1, secondary_y=True)
fig.update_layout(barmode="stack")
fig.update_xaxes(title_text="Date", row=len(ZONE_KEYS), col=1)
style_fig(fig, "Production mix and carbon intensity by region, daily mean", height=1150)
fig.update_yaxes(showgrid=False, secondary_y=True)  # avoid double gridlines
fig.show()

### 3. Production by source

Daily-mean production per source, one line per source, per region. Shows how each fuel's contribution moves over the five years (wind growth, nuclear outages, seasonal hydro and solar).

In [ ]:
fig = make_subplots(
    rows=len(ZONE_KEYS), cols=1, shared_xaxes=True,
    subplot_titles=ZONE_KEYS, vertical_spacing=0.03,
)
seen = set()
for i, zone in enumerate(ZONE_KEYS, start=1):
    daily = daily_prod_by_source(zone)
    for source in daily.columns:
        show = source not in seen
        seen.add(source)
        fig.add_trace(
            go.Scatter(x=daily.index, y=daily[source], mode="lines",
                       line=dict(color=ENERGY_PALETTE.get(source, "#888"), width=1),
                       name=source, legendgroup=source, showlegend=show),
            row=i, col=1,
        )
    fig.update_yaxes(title_text="MW", row=i, col=1)
fig.update_xaxes(title_text="Date", row=len(ZONE_KEYS), col=1)
style_fig(fig, "Production by source over time, daily mean by region", height=1150)
fig.show()

### 4. Carbon intensity over net cross-border flow

Daily-mean net imports (MW, left axis; positive = importing, negative = exporting) as semi-transparent bars, with carbon intensity (right axis) overlaid. Reveals whether a region leans on imports and whether import-heavy periods coincide with lower or higher CI.

In [ ]:
fig = make_subplots(
    rows=len(ZONE_KEYS), cols=1, shared_xaxes=True,
    subplot_titles=ZONE_KEYS, vertical_spacing=0.03,
    specs=[[{"secondary_y": True}] for _ in ZONE_KEYS],
)
for i, zone in enumerate(ZONE_KEYS, start=1):
    net = daily_net_flow(zone)
    fig.add_trace(
        go.Bar(x=net.index, y=net.values, name="net imports",
               marker_color="#3B6A96", marker_line_width=0, opacity=0.5,
               legendgroup="net", showlegend=(i == 1)),
        row=i, col=1, secondary_y=False,
    )
    ci = daily_ci(zone)
    fig.add_trace(
        go.Scatter(x=ci.index, y=ci.values, mode="lines",
                   line=dict(color="#C0504D", width=1),
                   name="carbon intensity", legendgroup="ci", showlegend=(i == 1)),
        row=i, col=1, secondary_y=True,
    )
    fig.update_yaxes(title_text="Net MW", row=i, col=1, secondary_y=False)
    fig.update_yaxes(title_text="gCO2eq/kWh", row=i, col=1, secondary_y=True)
fig.update_xaxes(title_text="Date", row=len(ZONE_KEYS), col=1)
style_fig(fig, "Net cross-border flow and carbon intensity by region, daily mean", height=1150)
fig.update_yaxes(showgrid=False, secondary_y=True)
fig.show()

# Section 2 — Distributions

How carbon intensity and the production mix are distributed: across the value range, across the calendar year, and across the daily and weekly cycle.

### 5. Carbon intensity distribution

Overlapping histograms, one per region, semi-transparent. Reveals the shape and spread of each region's CI: FI low and right-skewed, SG high and narrow, BE wide, the US zones in between.

In [ ]:
fig = px.histogram(
    ci_long,
    x="carbon_intensity_gco2eq_kwh",
    color="zone",
    color_discrete_map=REGIONAL_PALETTE,
    category_orders={"zone": ZONE_KEYS},
    barmode="overlay",
    opacity=0.55,
    nbins=80,
    labels={"carbon_intensity_gco2eq_kwh": "Carbon Intensity (gCO2eq/kWh)", "zone": "Zone"},
)
fig.update_yaxes(title_text="Count")
style_fig(fig, "Carbon intensity distribution by region")
fig.show()

### 6. Annual production mix, share of total

Each region's annual production normalised to 100 percent, so composition is comparable despite zone scales differing by an order of magnitude. Baseload (nuclear) at the bottom, fossils on top.

In [ ]:
records = []
for zone in ZONE_KEYS:
    pb = load_pb(zone)
    if pb.empty:
        continue
    prod_cols = [c for c in pb.columns if c.startswith("prod_") and c.endswith("_mw")]
    src = pb[prod_cols].copy()
    src.columns = [c[len("prod_"):-len("_mw")] for c in prod_cols]
    annual = src.groupby(src.index.year).mean()
    for year in annual.index:
        for source in annual.columns:
            val = annual.loc[year, source]
            if pd.notna(val):
                records.append({"zone": zone, "year": int(year), "source": source, "mw": val})
long_prod = pd.DataFrame(records)
long_prod["pct"] = (
    100.0 * long_prod["mw"]
    / long_prod.groupby(["zone", "year"])["mw"].transform("sum")
)

fig = px.bar(
    long_prod,
    x="year", y="pct", color="source",
    facet_col="zone", facet_col_spacing=0.04,
    color_discrete_map=ENERGY_PALETTE,
    category_orders={"source": list(ENERGY_SOURCE_ORDER), "zone": ZONE_KEYS},
    labels={"pct": "Share of Production (%)", "year": "Year", "source": "Source"},
)
fig.update_xaxes(tickmode="linear", dtick=1)
fig.update_yaxes(range=[0, 100], ticksuffix="%")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(barmode="stack")
style_fig(fig, "Annual production mix per zone, share of total, 2021 to 2025", width=int(PLOT_W * 1.6))
fig.show()

### 7. Carbon intensity by day of week and hour of day

One heatmap per region (stacked vertically), hours of day as columns and days of week as rows, square cells. Each region is scaled to its own range so the within-region weekly and daily pattern is visible regardless of absolute CI level.

In [ ]:
DAY_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig = make_subplots(
    rows=len(ZONE_KEYS), cols=1,
    subplot_titles=ZONE_KEYS, vertical_spacing=0.06,
)
for i, zone in enumerate(ZONE_KEYS, start=1):
    s = ci_long[ci_long["zone"] == zone]["carbon_intensity_gco2eq_kwh"]
    g = (
        s.to_frame("ci")
        .assign(dow=s.index.dayofweek, hod=s.index.hour)
        .groupby(["dow", "hod"])["ci"].mean().unstack("hod")
    )
    g.index = [DAY_LABELS[d] for d in g.index]
    fig.add_trace(
        go.Heatmap(z=g.values, x=list(g.columns), y=list(g.index),
                   colorscale=MUTED_ORANGE, showscale=False),
        row=i, col=1,
    )
    fig.update_xaxes(title_text=("Hour of Day" if i == len(ZONE_KEYS) else None), row=i, col=1)
style_fig(fig, "Carbon intensity by day of week and hour of day, per region", width=800, height=1500)
# Square cells: lock each subplot y to its x at 1:1, Monday on top.
for i in range(1, len(ZONE_KEYS) + 1):
    xref = "x" if i == 1 else f"x{i}"
    fig.update_yaxes(scaleanchor=xref, scaleratio=1, autorange="reversed", row=i, col=1)
fig.show()

### 8. Carbon intensity by month and year

One heatmap per region (stacked vertically), months as columns and years as rows, square cells, same muted-orange scale. Reveals year-over-year drift and the seasonal band within each year.

In [ ]:
MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig = make_subplots(
    rows=len(ZONE_KEYS), cols=1,
    subplot_titles=ZONE_KEYS, vertical_spacing=0.06,
)
for i, zone in enumerate(ZONE_KEYS, start=1):
    s = ci_long[ci_long["zone"] == zone]["carbon_intensity_gco2eq_kwh"]
    g = (
        s.to_frame("ci")
        .assign(year=s.index.year, month=s.index.month)
        .groupby(["year", "month"])["ci"].mean().unstack("month")
    )
    fig.add_trace(
        go.Heatmap(z=g.values,
                   x=[MONTH_LABELS[m - 1] for m in g.columns],
                   y=[str(y) for y in g.index],
                   colorscale=MUTED_ORANGE, showscale=False),
        row=i, col=1,
    )
    fig.update_xaxes(title_text=("Month" if i == len(ZONE_KEYS) else None), row=i, col=1)
style_fig(fig, "Carbon intensity by month and year, per region", width=800, height=1500)
for i in range(1, len(ZONE_KEYS) + 1):
    xref = "x" if i == 1 else f"x{i}"
    fig.update_yaxes(scaleanchor=xref, scaleratio=1, autorange="reversed", row=i, col=1)
fig.show()

# Findings

(Fill in after reviewing the figures.)

**Section 1 — time series**
- Baseline CI:
- CI vs production mix:
- Production by source:
- CI vs net flow:

**Section 2 — distributions**
- CI distribution:
- Annual production mix:
- Day-of-week / hour-of-day:
- Month / year:

These feed the report's Data section and the per-region heterogeneity atlas (Contribution 3).